<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  
  <b>Module 2.4</b>  
  <span style="color:#9ca3af;">Metadata Management</span>
</div>


Metadata makes retrieval smarter. Every `Document` carries a `metadata` dict that can be used for **filtering, routing, and lineage tracking**.

Key metadata fields to add:
- `source` — file/URL path
- `page` — page number
- `chunk_id` — unique chunk identifier
- `author`, `created_at`, `version` — document lineage

In [1]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from datetime import datetime
import hashlib

# ── Metadata-enriched document pipeline ──────────────────────────────────────
def enrich_metadata(docs: list[Document], extra: dict) -> list[Document]:
    """Add extra metadata fields to every document in the list."""
    enriched = []
    for i, doc in enumerate(docs):
        meta = {
            **doc.metadata,
            **extra,
            "chunk_id"   : hashlib.md5(doc.page_content.encode()).hexdigest()[:8],
            "chunk_index": i,
            "ingested_at": datetime.utcnow().isoformat(),
        }
        enriched.append(Document(page_content=doc.page_content, metadata=meta))
    return enriched


raw_text = """Chapter 1: Introduction to Machine Learning
Machine learning is a subset of artificial intelligence that enables systems to learn from data.

Chapter 2: Supervised Learning
In supervised learning, models are trained on labelled data to make predictions.

Chapter 3: Unsupervised Learning
Unsupervised learning discovers hidden patterns in data without labels.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
raw_docs = splitter.create_documents([raw_text], metadatas=[{"source": "ml_textbook.pdf"}])

enriched_docs = enrich_metadata(raw_docs, {
    "author"  : "Jane Doe",
    "version" : "v2.1",
    "category": "textbook",
})

for d in enriched_docs:
    print(f"chunk_id={d.metadata['chunk_id']} | index={d.metadata['chunk_index']}")
    print(f"  {d.page_content[:80].strip()}")
    print(f"  metadata → {d.metadata}\n")


chunk_id=1b8778d8 | index=0
  Chapter 1: Introduction to Machine Learning
Machine learning is a subset of arti
  metadata → {'source': 'ml_textbook.pdf', 'author': 'Jane Doe', 'version': 'v2.1', 'category': 'textbook', 'chunk_id': '1b8778d8', 'chunk_index': 0, 'ingested_at': '2026-05-09T18:01:33.340825'}

chunk_id=9f0c8872 | index=1
  Chapter 2: Supervised Learning
In supervised learning, models are trained on lab
  metadata → {'source': 'ml_textbook.pdf', 'author': 'Jane Doe', 'version': 'v2.1', 'category': 'textbook', 'chunk_id': '9f0c8872', 'chunk_index': 1, 'ingested_at': '2026-05-09T18:01:33.340857'}

chunk_id=43770db7 | index=2
  Chapter 3: Unsupervised Learning
Unsupervised learning discovers hidden patterns
  metadata → {'source': 'ml_textbook.pdf', 'author': 'Jane Doe', 'version': 'v2.1', 'category': 'textbook', 'chunk_id': '43770db7', 'chunk_index': 2, 'ingested_at': '2026-05-09T18:01:33.340870'}



C:\Users\mohdf\AppData\Local\Temp\ipykernel_6784\3758703525.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow().isoformat(),


In [2]:
# ── Metadata filtering in retrieval ──────────────────────────────────────────
embeddings  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(enriched_docs, embeddings, collection_name="meta_demo")

# Filter: only return chunks from version v2.1 authored by Jane Doe
filter_results = vectorstore.similarity_search(
    "What is supervised learning?",
    k=2,
    filter={"author": "Jane Doe"}
)

print("Filtered retrieval results:")
for r in filter_results:
    print(f"  [{r.metadata['chunk_id']}] {r.page_content[:80].strip()}")
    print(f"  Author: {r.metadata['author']} | Version: {r.metadata['version']}\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Filtered retrieval results:
  [9f0c8872] Chapter 2: Supervised Learning
In supervised learning, models are trained on lab
  Author: Jane Doe | Version: v2.1

  [1b8778d8] Chapter 1: Introduction to Machine Learning
Machine learning is a subset of arti
  Author: Jane Doe | Version: v2.1

